In [ ]:
pip show autogen_core

Name: autogen-core
Version: 0.7.5
Summary: Foundational interfaces and agent runtime implementation for AutoGen
Home-page: 
Author: 
Author-email: 
License: MIT License

    Copyright (c) Microsoft Corporation.

    Permission is hereby granted, free of charge, to any person obtaining a copy
    of this software and associated documentation files (the "Software"), to deal
    in the Software without restriction, including without limitation the rights
    to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
    copies of the Software, and to permit persons to whom the Software is
    furnished to do so, subject to the following conditions:

    The above copyright notice and this permission notice shall be included in all
    copies or substantial portions of the Software.

    THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
    IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
    FITNESS FOR A PARTICULAR PURPOSE A

In [ ]:
import asyncio
import json
import heapq
from dataclasses import dataclass, field
from enum import Enum, IntEnum
from typing import List, Tuple, Dict, Optional
from autogen_core import RoutedAgent, message_handler, SingleThreadedAgentRuntime, AgentId, DefaultTopicId, TypePrefixSubscription
from pydantic import BaseModel

In [ ]:
class Deposito(BaseModel):
    id: str
    posicion: Tuple[int, int]
    paquetes_pares: int
    paquetes_impares: int

class Cliente(BaseModel):
    id: str
    posicion: Tuple[int, int]
    prioridad: str
    demandas: Dict[str, int]
    recibido: Dict[str, int] = {}

class VehiculoEntorno(BaseModel):
    id: str
    posicion: Tuple[int, int]
    capacidad_max: int
    carga_actual: int = 0
    coste_paso: int=1
    metricas: Dict[str, int] = {
        "desplazamientos": 0,
        "coste_total": 0,
        "espacio_desperdiciado": 0,
        "capacidad_ociosa": 0
    }

class EstadoEntorno(BaseModel):
    mapa: List[List[str]]
    depositos: List[Deposito]
    clientes: List[Cliente]
    vehiculos: List[VehiculoEntorno]
    dimensiones: Tuple[int, int]
    penalizacion_trafico: int = 3

In [ ]:
class Prioridad(IntEnum):
    ALTA = 3
    MEDIA = 2
    BAJA = 1

class Celda(str, Enum):
    LIBRE = "."
    OBSTACULO = "#"
    DEPOSITO = "D"
    CLIENTE = "C"
    TRAFICO = "T"

In [ ]:
class SolicitudSuministro(BaseModel): # Enviado por Cliente -> Coordinador
    id_cliente: str
    posicion_entrega: Tuple[int, int]
    prioridad: Prioridad
    articulos: Dict[str, int] # {id_deposito: cantidad}

class CallForProposal(BaseModel):
    id_subasta: str
    id_cliente: str
    pos_cliente: Tuple[int, int]
    id_deposito: str
    pos_deposito: Tuple[int, int]
    cantidad: int
    prioridad: Prioridad

class Adjudicacion(BaseModel):
    id_subasta: str
    aceptado: bool
    id_deposito: Optional[str] = None
    pos_deposito: Optional[Tuple[int, int]] = None # 
    id_cliente: Optional[str] = None
    pos_cliente: Optional[Tuple[int, int]] = None
    cantidad: Optional[int] = None

class Proposal(BaseModel):
    id_subasta: str
    id_vehiculo: str
    coste_estimado: float
    puede_realizar: bool
    capacidad_libre_tras_tarea: int
    cantidad_ofrecida: int

class ConsultaPosicion(BaseModel): # Vehículo -> Depósito
    id_vehiculo: str

class RespuestaPosicion(BaseModel): # Depósito -> Vehículo
    id_deposito: str
    posicion: Tuple[int, int]

class MensajeInicio(BaseModel):
    comando: str

class PeticionCarga(BaseModel):
    volumen_solicitado: int

class EntregaMercancia(BaseModel):
    id_vehiculo: str
    id_deposito_origen: str
    cantidad: int

class RespuestaCarga(BaseModel):
    id_deposito: str
    pares_entregados: int
    impares_entregados: int
    pares_restantes: int      # Lo que le queda al depósito
    impares_restantes: int    # Lo que le queda al depósito
    volumen_solicitado_original: int # Para que el vehículo ajuste sus reservas

In [ ]:
class AgenteDeposito(RoutedAgent):
    def __init__(self, id, posicion, pares, impares):
        super().__init__(id)
        self.posicion = posicion
        self.paquetes_pares = pares
        self.paquetes_impares = impares

    @message_handler
    async def procesar_peticion_carga(self, message: PeticionCarga, ctx) -> None:
        """El depósito recibe la petición, procesa su stock y envía lo que ha podido cargar."""
        vol_pendiente = message.volumen_solicitado
        
        # 1. Prioridad: Paquetes pares (ocupan 2 uds de volumen)
        pares_a_dar = min(vol_pendiente // 2, self.paquetes_pares)
        self.paquetes_pares -= pares_a_dar
        vol_pendiente -= (pares_a_dar * 2)
        
        # 2. Rellenar con paquetes impares (ocupan 1 ud)
        impares_a_dar = min(vol_pendiente, self.paquetes_impares)
        self.paquetes_impares -= impares_a_dar
        
        # 3. Enviar respuesta al vehículo con la mercancía y el stock restante
        respuesta = RespuestaCarga(
            id_deposito=self.id.type,
            pares_entregados=pares_a_dar,
            impares_entregados=impares_a_dar,
            pares_restantes=self.paquetes_pares,     # Manda sus restantes
            impares_restantes=self.paquetes_impares, # Manda sus restantes
            volumen_solicitado_original=message.volumen_solicitado
        )
        
        await self.send_message(respuesta, recipient=ctx.sender)

In [ ]:
class AgenteCoordinador(RoutedAgent):
    def __init__(self, id, num_clientes, num_vehiculos, depositos_info, upper_limit, low_limit):
        super().__init__(id)
        self.num_clientes = num_clientes
        self.num_vehiculos = num_vehiculos
        self.peticiones_pendientes = []
        self.respuestas_lote = []
        self.indice_subasta_actual = 0
        self.tareas_ordenadas = []
        self.mapa_depositos = depositos_info
        self.fase_subasta_completa = False

        # --- VARIABLES DE APRENDIZAJE POR REFUERZO (Q-LEARNING) ---
        self.q_file = "q_table.json"
        self.q_table = self._cargar_q_table()
        self.epsilon = 0.2  # 20% de las veces explora acciones aleatorias
        self.alpha = 0.1    # Tasa de aprendizaje (qué tanto caso hace a la nueva experiencia)
        self.gamma = 0.9    # Factor de descuento (visión a largo plazo)
        self.upper_limit = upper_limit  # Para normalizar recompensas
        self.low_limit = low_limit
    def _cargar_q_table(self):
        """Carga la memoria del agente desde un archivo. Si no existe, nace sin conocimiento."""
        if os.path.exists(self.q_file):
            with open(self.q_file, "r") as f:
                return json.load(f)
        # Estados: ALTA demanda (>10), MEDIA (6-10), BAJA (1-5)
        # Acciones: 0 (Priorizar Cantidad), 1 (Priorizar Coste), 2 (Equilibrado)
        return {"ALTA": [0.0, 0.0, 0.0], "MEDIA": [0.0, 0.0, 0.0], "BAJA": [0.0, 0.0, 0.0]}

    def _guardar_q_table(self):
        """Guarda lo aprendido para futuras simulaciones."""
        with open(self.q_file, "w") as f:
            json.dump(self.q_table, f, indent=4)

    def _obtener_estado(self, cantidad):
        """Discretiza el entorno en 3 estados según el tamaño del pedido."""
        if cantidad > self.upper_limit: 
            return "ALTA"
        elif cantidad > self.low_limit: 
            return "MEDIA"
        return "BAJA"

    def _elegir_accion(self, estado):
        """Política Epsilon-Greedy: Elige la mejor acción conocida o explora una nueva."""
        if random.uniform(0, 1) < self.epsilon:
            return random.choice([0, 1, 2]) # Exploración
        else:
            valores = self.q_table[estado]
            return valores.index(max(valores)) # Explotación

    @message_handler
    async def iniciar_proceso_logistico(self, message: MensajeInicio, ctx) -> None:
        """El orquestador despierta a todos los clientes."""
        if message.comando == "START":
            print("📢 Coordinador: Iniciando fase de descubrimiento de demanda...")
            for i in range(1, self.num_clientes + 1):
                await self.send_message(MensajeInicio(comando="START"), AgentId(f"C{i}", "default"))
                
        elif message.comando == "REVISAR_PENDIENTES":
            # Filtramos solo las tareas que aún tienen cantidad por entregar
            pendientes = [t for t in self.tareas_ordenadas if t["cantidad"] > 0]
            
            # Si hay pendientes y la ronda anterior de subastas ya terminó
            if pendientes and self.fase_subasta_completa:
                self.tareas_ordenadas = pendientes
                self.indice_subasta_actual = 0
                self.fase_subasta_completa = False
                await self.iniciar_siguiente_subasta()

    @message_handler
    async def recibir_solicitudes_iniciales(self, message: SolicitudSuministro, ctx) -> None:
        """Fase 1: Recopilar todas las solicitudes iniciales."""
        self.peticiones_pendientes.append(message) 

        # Una vez recibidas todas las de los clientes, ordenamos y empezamos
        if len(self.peticiones_pendientes) == self.num_clientes:
            print("📦 Coordinador: Todas las peticiones recibidas. Ordenando por prioridad...")
            self._preparar_y_ordenar_lote()
            await self.iniciar_siguiente_subasta()

    def _preparar_y_ordenar_lote(self):
        """Ordenación: 1º Prioridad (desc), 2º Cantidad (desc)."""
        for sol in self.peticiones_pendientes:
            for dep_id, cant in sol.articulos.items():
                self.tareas_ordenadas.append({
                    "id_subasta": f"sub_{sol.id_cliente}_{dep_id}",
                    "prioridad": sol.prioridad,
                    "cantidad": cant,
                    "id_cliente": sol.id_cliente,
                    "pos_cliente": sol.posicion_entrega,
                    "id_deposito": dep_id,
                    "pos_deposito": self.mapa_depositos.get(dep_id),
                })
        # Aplicamos el criterio de ordenación doble
        self.tareas_ordenadas.sort(key=lambda x: (-x["prioridad"].value, -x["cantidad"]))
    
    async def iniciar_siguiente_subasta(self):
        """Fase 2: Lanzar CFP para la tarea actual del lote."""
        if self.indice_subasta_actual < len(self.tareas_ordenadas):
            tarea = self.tareas_ordenadas[self.indice_subasta_actual]            
            self.respuestas_lote = []
            cfp = CallForProposal(
                id_subasta=tarea["id_subasta"],
                id_cliente=tarea["id_cliente"],
                pos_cliente=tarea["pos_cliente"],
                id_deposito=tarea["id_deposito"],
                pos_deposito=tarea["pos_deposito"],
                cantidad=tarea["cantidad"],
                prioridad=tarea["prioridad"]
            )
            await self.publish_message(cfp, topic_id=DefaultTopicId())
        else:
            print("🏁 Lote de subastas finalizado.")
            self.fase_subasta_completa = True

    @message_handler
    async def gestionar_propuestas(self, message: Proposal, ctx) -> None:
        """Fase 3: Evaluar Propose/Refuse y adjudicar."""
        self.respuestas_lote.append(message)
        
        if len(self.respuestas_lote) == self.num_vehiculos:
            await self.seleccionar_ganador_y_continuar()
    
    async def seleccionar_ganador_y_continuar(self):
        tarea = self.tareas_ordenadas[self.indice_subasta_actual]
        validas = [r for r in self.respuestas_lote if r.puede_realizar]
        estado_s = self._obtener_estado(tarea["cantidad"])
        accion_elegida = self._elegir_accion(estado_s)
        if accion_elegida == 0:   
            pesos = (0.7, 0.2, 0.1) # Acción 0: Ansioso por cargar cantidad
        elif accion_elegida == 1: 
            pesos = (0.3, 0.6, 0.1) # Acción 1: Tacaño con los costes
        else:               
            pesos = (0.4, 0.4, 0.2) # Acción 2: Equilibrado
        if validas:
            max_coste = max((p.coste_estimado for p in validas), default=1) or 1
            max_cap = max((p.capacidad_libre_tras_tarea for p in validas), default=1) or 1

            def calcular_puntuacion(p):
                f_cantidad = 1.0 - (p.cantidad_ofrecida / tarea["cantidad"]) 
                f_coste = p.coste_estimado / max_coste 
                f_desperdicio = p.capacidad_libre_tras_tarea / max_cap 
                return (pesos[0] * f_cantidad) + (pesos[1] * f_coste) + (pesos[2] * f_desperdicio)

            ganador = min(validas, key=calcular_puntuacion)
            recompensa = (ganador.cantidad_ofrecida * 5) - ganador.coste_estimado

            print(f"🏆 Ganador para subasta {tarea['id_subasta']}: {ganador.id_vehiculo}")
            print(f"   (Coste marginal: {ganador.coste_estimado}, Espacio restante: {ganador.capacidad_libre_tras_tarea})")

            for resp in self.respuestas_lote:
                es_ganador = (resp.id_vehiculo == ganador.id_vehiculo)
                cantidad_adjudicada = ganador.cantidad_ofrecida if es_ganador else 0
                await self.send_message(
                    Adjudicacion(
                        id_subasta=tarea["id_subasta"], aceptado=es_ganador,
                        id_deposito=tarea["id_deposito"], 
                        pos_deposito=tarea["pos_deposito"], 
                        cantidad=cantidad_adjudicada,
                        id_cliente=tarea["id_cliente"], pos_cliente=tarea["pos_cliente"]
                    ),
                    recipient=AgentId(resp.id_vehiculo, "default")
                )
            tarea["cantidad"] -= ganador.cantidad_ofrecida
            cantidad_restante = tarea["cantidad"]
            if cantidad_restante > 0:
                # En la siguiente ronda, el coordinador volverá a lanzar un CFP por el pico restante.
                print(f"🔄 Pedido fragmentado: Quedan {tarea['cantidad']} uds pendientes para {tarea['id_cliente']}. Re-subastando el resto...")
            else:
                # Pedido cubierto al 100%, pasamos a la siguiente subasta
                self.indice_subasta_actual += 1
            estado_s_prima = self._obtener_estado(cantidad_restante) if cantidad_restante > 0 else (
                self._obtener_estado(self.tareas_ordenadas[self.indice_subasta_actual]["cantidad"]) 
                if self.indice_subasta_actual < len(self.tareas_ordenadas) else "BAJA"
            )
            
            q_actual = self.q_table[estado_s][accion_a]
            max_q_siguiente = max(self.q_table[estado_s_prima])
            # Fórmula principal del Q-Learning
            nuevo_q = q_actual + self.alpha * (recompensa + self.gamma * max_q_siguiente - q_actual)
            self.q_table[estado_s][accion_a] = nuevo_q
            self._guardar_q_table()
            print(f"🧠 RL Update -> S:[{estado_s}] A:[{accion_a}] Recompensa: {recompensa:.1f} | Nuevo Q-Valor: {nuevo_q:.2f}")
            
            

        else:
            print(f"⚠️ Ningún vehículo pudo asumir la tarea {tarea['id_subasta']}")
            recompensa = -10 
            q_actual = self.q_table[estado_s][accion_a]
            self.q_table[estado_s][accion_a] = q_actual + self.alpha * (recompensa - q_actual)
            self._guardar_q_table()
            self.indice_subasta_actual += 1
        await self.iniciar_siguiente_subasta()

In [ ]:
class AgenteVehiculoCoordinado(RoutedAgent):
    def __init__(self, id, capacidad, coste_paso, penalizacion_t, posicion_inicial):
        super().__init__(id)
        self.capacidad_max = capacidad
        self.coste_paso = coste_paso
        self.posicion = posicion_inicial
        self.penalizacion_t = penalizacion_t
        
        self.carga_actual = 0
        self.capacidad_reservada = 0
        self.inventario_interno = {}      
          
        self.plan_paradas_futuras = [] 
        self.plan_ruta_pasos = []      
        self.mapa_memoria = None       
        self.cache_distancias = {}

        self.metricas = {
            "desplazamientos": 0, "coste_total": 0,
            "espacio_desperdiciado": 0, "capacidad_ociosa": 0
        }

    def _simular_costo_plan(self, plan):
        if not plan: return 0 
        costo_total = 0
        punto_actual = self.posicion
        for _, pos_destino, _, _ , _ in plan:
            costo_total += self.estimar_coste_A_estrella(punto_actual, pos_destino)
            punto_actual = pos_destino
        return costo_total

    def _es_plan_valido(self, plan):
        carga_simulada = self.carga_actual
        for tipo, _, _, cant, _ in plan:
            if tipo == "RECOGER":
                carga_simulada += cant
            elif tipo == "ENTREGAR":
                carga_simulada -= cant
            if carga_simulada > self.capacidad_max:
                return False
        return True

    @message_handler
    async def evaluar_cfp(self, message: CallForProposal, ctx) -> None:
        espacio_libre = self.capacidad_max - self.carga_actual - self.capacidad_reservada
        
        # FIX 1: Retornar tras rechazar la subasta para no enviar propuestas extra
        if espacio_libre <= 0:
            await self.send_message(Proposal(id_subasta=message.id_subasta, id_vehiculo=self.id.type, cantidad_ofrecida=0,
                                            coste_estimado=float('inf'), capacidad_libre_tras_tarea=0, puede_realizar=False), ctx.sender)
            return

        cantidad_a_ofrecer = min(espacio_libre, message.cantidad)
        costo_actual = self._simular_costo_plan(self.plan_paradas_futuras)
        mejor_incremento = float('inf')
        
        nueva_recogida = ("RECOGER", message.pos_deposito, message.id_deposito, cantidad_a_ofrecer, message.id_deposito)
        nueva_entrega = ("ENTREGAR", message.pos_cliente, message.id_cliente, cantidad_a_ofrecer, message.id_deposito)

        n = len(self.plan_paradas_futuras)
        for i in range(n + 1):
            for j in range(i + 1, n + 2):
                plan_test = list(self.plan_paradas_futuras)
                plan_test.insert(i, nueva_recogida)
                plan_test.insert(j, nueva_entrega)
                if self._es_plan_valido(plan_test):
                    incremento = self._simular_costo_plan(plan_test) - costo_actual
                    if incremento < mejor_incremento:
                        mejor_incremento = incremento

        if mejor_incremento != float('inf'):
            await self.send_message(Proposal(id_subasta=message.id_subasta, id_vehiculo=self.id.type,
                coste_estimado=mejor_incremento, capacidad_libre_tras_tarea=(espacio_libre - cantidad_a_ofrecer), puede_realizar=True, cantidad_ofrecida = cantidad_a_ofrecer), ctx.sender)

    @message_handler
    async def recibir_adjudicacion(self, message: Adjudicacion, ctx) -> None:
        if message.aceptado:
            self.capacidad_reservada += message.cantidad
            pos_dep = message.pos_deposito
            
            nueva_recogida = ("RECOGER", message.pos_deposito, message.id_deposito, message.cantidad, message.id_deposito)
            nueva_entrega = ("ENTREGAR", message.pos_cliente, message.id_cliente, message.cantidad, message.id_deposito)
            
            mejor_i, mejor_j = 0, 1
            min_costo = float('inf')
            
            n = len(self.plan_paradas_futuras)
            for i in range(n + 1):
                for j in range(i + 1, n + 2):
                    plan_test = list(self.plan_paradas_futuras)
                    plan_test.insert(i, nueva_recogida)
                    plan_test.insert(j, nueva_entrega)
                    if self._es_plan_valido(plan_test):
                        c = self._simular_costo_plan(plan_test)
                        if c < min_costo:
                            min_costo = c
                            mejor_i, mejor_j = i, j

            # FIX 2: Reconstruir plan_test con las mejores posiciones reales antes de consolidar
            plan_test = list(self.plan_paradas_futuras)
            plan_test.insert(mejor_i, nueva_recogida)
            plan_test.insert(mejor_j, nueva_entrega)

            plan_consolidado = []
            for parada in plan_test:
                if not plan_consolidado:
                    plan_consolidado.append(parada)
                else:
                    ultima = plan_consolidado[-1]
                    if ultima[0] == parada[0] and ultima[1] == parada[1] and ultima[2] == parada[2] and ultima[4] == parada[4]:
                        plan_consolidado[-1] = (ultima[0], ultima[1], ultima[2], ultima[3] + parada[3], ultima[4])
                    else: 
                        plan_consolidado.append(parada)
                        
            self.plan_paradas_futuras = plan_consolidado
            self.plan_ruta_pasos = [] 
            print(f"🏆 {self.id.type} ganó {message.id_subasta} (Inserción en pos {mejor_i} y {mejor_j})")
    
    @message_handler
    async def actualizar_y_decidir(self, message: EstadoEntorno, ctx) -> None:
        self.mapa_memoria = message.mapa 

        if not self.plan_ruta_pasos and self.plan_paradas_futuras:
            tipo, destino, _, _ , _ = self.plan_paradas_futuras[0]
            if self.posicion == destino:
                await self.verificar_llegada_coordinada(message) 
            else:
                self.plan_ruta_pasos = self.calcular_a_estrella(self.posicion, destino, message.mapa)
        
        if self.plan_ruta_pasos:
            self.posicion = self.plan_ruta_pasos.pop(0)
            self.metricas["desplazamientos"] += 1
            self._sincronizar_con_entorno(message) 
            coste_paso_actual = self.coste_paso
            if self.mapa_memoria[self.posicion[0]][self.posicion[1]] == 'T':
                coste_paso_actual += self.penalizacion_t
            self.metricas["coste_total"] += coste_paso_actual
            self.metricas["espacio_desperdiciado"] += (self.capacidad_max - self.carga_actual)
            
            if not self.plan_ruta_pasos:
                await self.verificar_llegada_coordinada(message)
        else:
            self.metricas["capacidad_ociosa"] += self.capacidad_max
            self._sincronizar_con_entorno(message)

    async def verificar_llegada_coordinada(self, estado):
        if not self.plan_paradas_futuras: 
            return
        tipo, pos, entidad_id, cant, id_origen = self.plan_paradas_futuras.pop(0)        
        if tipo == "RECOGER":
            await self.send_message(PeticionCarga(volumen_solicitado=cant), recipient=AgentId(id_origen, "default"))
        elif tipo == "ENTREGAR":
            self.carga_actual -= cant
            self.inventario_interno[entidad_id] = 0
            
            # FIX 3: Sincronizar la entrega con el estado del mapa para que el visualizador muestre los recibidos
            for c in estado.clientes:
                if c.id == entidad_id:
                    c.recibido[id_origen] = c.recibido.get(id_origen, 0) + cant

            await self.send_message(EntregaMercancia(id_vehiculo=self.id.type, id_deposito_origen=id_origen, cantidad=cant), recipient=AgentId(entidad_id, "default"))
            print(f"🚚 {self.id.type} descargó {cant} uds y notificó la entrega a {entidad_id}")
            
    @message_handler
    async def recibir_carga(self, message: RespuestaCarga, ctx) -> None:
        volumen_real_recibido = (message.pares_entregados * 2) + message.impares_entregados
        self.carga_actual += volumen_real_recibido
        self.capacidad_reservada -= message.volumen_solicitado_original
        self.inventario_interno[message.id_deposito] = self.inventario_interno.get(message.id_deposito, 0) + volumen_real_recibido
        
        print(f"📦 {self.id.type} cargó de {message.id_deposito}: {message.pares_entregados} pares y {message.impares_entregados} impares.")

    def estimar_coste_A_estrella(self, inicio, fin):
        if inicio == fin : return 0
        tupla_ruta = (inicio, fin)
        if tupla_ruta in self.cache_distancias: return self.cache_distancias[tupla_ruta]
        ruta = self.calcular_a_estrella(inicio, fin, self.mapa_memoria)
        coste = 0
        for r, c in ruta:
            coste += self.coste_paso
            if self.mapa_memoria[r][c] == 'T': coste += self.penalizacion_t
        self.cache_distancias[tupla_ruta] = coste
        self.cache_distancias[(fin, inicio)] = coste 
        return coste

    def calcular_a_estrella(self, inicio, fin, mapa):
        if not mapa or inicio == fin: return []
        direcciones = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        open_list = [(0, inicio)]
        camino_previo = {}
        coste_acumulado = {inicio: 0}
        while open_list:
            _, actual = heapq.heappop(open_list)
            if actual == fin:
                camino = []
                while actual in camino_previo:
                    camino.append(actual)
                    actual = camino_previo[actual]
                return camino[::-1]
            for dx, dy in direcciones:
                vecino = (actual[0] + dx, actual[1] + dy)
                if 0 <= vecino[0] < len(mapa) and 0 <= vecino[1] < len(mapa[0]) and mapa[vecino[0]][vecino[1]] != '#':
                    coste = self.coste_paso + (self.penalizacion_t if mapa[vecino[0]][vecino[1]] == 'T' else 0)
                    nuevo_coste = coste_acumulado[actual] + coste
                    if vecino not in coste_acumulado or nuevo_coste < coste_acumulado[vecino]:
                        coste_acumulado[vecino] = nuevo_coste
                        prioridad = nuevo_coste + max(abs(vecino[0]-fin[0]), abs(vecino[1]-fin[1]))
                        heapq.heappush(open_list, (prioridad, vecino))
                        camino_previo[vecino] = actual
        return []
        
    def _sincronizar_con_entorno(self, message):
        for v in message.vehiculos:
            if v.id == self.id.type:
                v.posicion = self.posicion
                v.carga_actual = self.carga_actual
                v.metricas = self.metricas

Task was destroyed but it is pending!
task: <Task pending name='Task-68' coro=<RunContext._run() running at d:\OneDrive\Escritorio\MASTER  ISIA\sma\venv\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py:110> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-567' coro=<RunContext._run() running at d:\OneDrive\Escritorio\MASTER  ISIA\sma\venv\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py:110> wait_for=<Future pending cb=[Task.task_wakeup()]>>


In [ ]:
class AgenteCliente(RoutedAgent):
    def __init__(self, id, posicion, demandas, prioridad):
        super().__init__(id)
        self.posicion = posicion
        self.demandas = demandas
        self.prioridad = prioridad
        self.recibido = {}

    @message_handler
    async def solicitar_ayuda(self, message:MensajeInicio , ctx) -> None:
        # El cliente levanta la mano si tiene demanda pendiente
        pendiente = {d: cant - self.recibido.get(d, 0) for d, cant in self.demandas.items() if cant > self.recibido.get(d, 0)}
        
        await self.send_message(
                SolicitudSuministro(id_cliente=self.id.type, posicion_entrega=self.posicion, 
                                   prioridad=self.prioridad, articulos=pendiente),
                recipient=AgentId("coordinador", "default")
        )

    @message_handler
    async def recibir_mercancia(self, message: EntregaMercancia, ctx) -> None:
        """El cliente procesa la mercancía recibida y actualiza su estado."""
        origen = message.id_deposito_origen
        cant = message.cantidad
        
        # 1. Actualiza su memoria interna
        self.recibido[origen] = self.recibido.get(origen, 0) + cant

        print(f"🤝 [{self.id.type}] Confirmada recepción de {cant} uds del depósito {origen} entregadas por {message.id_vehiculo}.")
        print(f"   📊 Estado {self.id.type}: Recibido {self.recibido} / Demanda {self.demandas}")


Ejemplo uso

In [ ]:
N, M = 10, 10
mapa_base = [['.' for _ in range(M)] for _ in range(N)]
for obs in [[1, 2], [1, 3], [2, 2], [3,3],[7,9], [4,4], [6,8]]: mapa_base[obs[0]][obs[1]] = '#'
for tr in [[4, 5], [4, 6], [4,7]]: mapa_base[tr[0]][tr[1]] = 'T'

# Crear el estado inicial
estado_inicial = EstadoEntorno(
    mapa=mapa_base,
    dimensiones=(N, M),
    penalizacion_trafico=3,
    depositos=[
        Deposito(id="D1", posicion=(0, 0), paquetes_pares=30, paquetes_impares=20),
        Deposito(id="D2", posicion=(7, 3), paquetes_pares=40, paquetes_impares=15)
    ],
    clientes=[
        Cliente(id="C1", posicion=(8, 8), prioridad="alta", demandas={"D1": 15, "D2": 25}, recibido={}),
        Cliente(id="C2", posicion=(2, 6), prioridad="media", demandas={"D1": 0, "D2": 10}, recibido={}),
        Cliente(id="C3", posicion=(2, 8), prioridad="media", demandas={"D1": 20, "D2": 15}, recibido={}),
        Cliente(id="C4", posicion=(4, 9), prioridad="alta", demandas={"D1": 15, "D2": 20}, recibido={}),
        Cliente(id="C5", posicion=(6, 9), prioridad="media", demandas={"D1": 15, "D2": 15}, recibido={}),
        Cliente(id="C6", posicion=(0, 6), prioridad="media", demandas={"D1": 15, "D2": 10}, recibido={})

    ],
    vehiculos=[
        VehiculoEntorno(id="V1", posicion=(4, 4), capacidad_max = 25),
        VehiculoEntorno(id="V2", posicion=(5, 5), capacidad_max = 20),
        VehiculoEntorno(id="V3", posicion=(3, 3), capacidad_max = 30)
    ]
)

In [ ]:
def mostrar_metricas_finales(estado_final, iteraciones_totales):
    
    # Usamos estado_final.vehiculos en lugar de lista_agentes
    total_desplazamientos = sum(v.metricas["desplazamientos"] for v in estado_final.vehiculos)
    total_coste = sum(v.metricas["coste_total"] for v in estado_final.vehiculos)
    total_desperdicio = sum(v.metricas["espacio_desperdiciado"] for v in estado_final.vehiculos)
    total_ociosa = sum(v.metricas["capacidad_ociosa"] for v in estado_final.vehiculos)

    print("\n" + "="*50)
    print("📊 MÉTRICAS FINALES DE LA SIMULACIÓN 📊")
    print("="*50)
    # Tiempo total (número de iteraciones) de la simulación [cite: 54]
    print(f"⏱️  Tiempo total (Iteraciones)    : {iteraciones_totales}")
    # Número de desplazamientos [cite: 55]
    print(f"🚚 Número de desplazamientos     : {total_desplazamientos}")
    # Coste total asociado a los vehículos [cite: 57]
    print(f"💰 Coste total asociado          : {total_coste} uds")
    # Espacio total desperdiciado por los vehículos en cada ruta [cite: 56]
    print(f"💨 Espacio desperdiciado en ruta : {total_desperdicio} uds de volumen")
    # Capacidad total de los vehículos que no están en ruta [cite: 58]
    print(f"💤 Capacidad ociosa (en espera)  : {total_ociosa} uds de volumen")
    print("="*50)

In [ ]:
import os

def mostrar_mapa_notebook(estado, iteracion):
    N, M = estado.dimensiones
    print(f"=== 🚚 SIMULACIÓN LOGÍSTICA - ITERACIÓN {iteracion} ===")
    
    grid_visual = [fila[:] for fila in estado.mapa]
    for d in estado.depositos: grid_visual[d.posicion[0]][d.posicion[1]] = f"[{d.id}]"
    for c in estado.clientes: grid_visual[c.posicion[0]][c.posicion[1]] = f"[{c.id}]"
    for v in estado.vehiculos:
        x, y = v.posicion
        if grid_visual[x][y] in ['.', '#', 'T'] or not grid_visual[x][y].startswith('['):
            grid_visual[x][y] = f"V{v.id[-1]}"
        else:
            grid_visual[x][y] += f",V{v.id[-1]}"

    for fila in grid_visual:
        print(" ".join(str(celda).ljust(8) for celda in fila))
        
    # Mostrar estado de los clientes
    print("-" * 40)
    for c in estado.clientes:
        print(f"Cliente {c.id} ({c.prioridad}) -> Demanda: {c.demandas} | Recibido: {c.recibido}")

In [ ]:
async def ejecutar_simulacion_coordinada(runtime, estado_actual, coordinador_id):
    # 1. Despertar al coordinador para que inicie las subastas
    print("🚀 FASE 1: INICIANDO SUBASTAS Y ASIGNACIÓN...")
    await runtime.send_message(MensajeInicio(comando="START"), recipient=coordinador_id)
    
    # Damos tiempo a que se resuelvan todas las subastas asíncronas
    await asyncio.sleep(5) 
    
    print("\n🚀 FASE 2: INICIANDO MOVIMIENTO FÍSICO...")
    iteracion = 0
    simulacion_terminada = False
    
    # Reutilizamos tu función de dibujado del entorno no coordinado
    mostrar_mapa_notebook(estado_actual, iteracion)

    while not simulacion_terminada and iteracion < 100:
        iteracion += 1

        # 1. En cada iteración, el coordinador intenta subastar lo que falte
        await runtime.send_message(MensajeInicio(comando="REVISAR_PENDIENTES"), recipient=coordinador_id)
        await asyncio.sleep(0.5) # Breve pausa para que los vehículos pujen si están libres
        # ------------------------------------
        # Enviar el estado a cada vehículo para que ejecuten su paso (A*)
        for v in estado_actual.vehiculos:
            agente_id = AgentId(type=v.id, key="default") 
            await runtime.send_message(message=estado_actual, recipient=agente_id)
            
        
        mostrar_mapa_notebook(estado_actual, iteracion)
        await asyncio.sleep(5)

        # Comprobar si todos los clientes están satisfechos
        mercancia_entregada = True
        for cliente in estado_actual.clientes:
            for dep_id, demanda in cliente.demandas.items():
                if cliente.recibido.get(dep_id, 0) < demanda:
                    mercancia_entregada = False
                    break
        
        if mercancia_entregada:
            simulacion_terminada = True
            print(f"\n🎉 ¡SIMULACIÓN COORDINADA COMPLETADA EN {iteracion} ITERACIONES!")

    # Imprimir métricas finales
    mostrar_metricas_finales(estado_actual, iteracion)

In [ ]:
async def setup_y_ejecutar():
    runtime = SingleThreadedAgentRuntime()

    # 1. Registrar Depósitos
    for d in estado_inicial.depositos:
        await AgenteDeposito.register(runtime, type=d.id, factory=lambda d_data=d: AgenteDeposito(d_data.id, d_data.posicion, d_data.paquetes_pares, d_data.paquetes_impares))

    # 2. Registrar Clientes
    for c in estado_inicial.clientes:
        await AgenteCliente.register(runtime, type=c.id, factory=lambda c_data=c: AgenteCliente(c_data.id, c_data.posicion, c_data.demandas, Prioridad.ALTA if c_data.prioridad == "alta" else Prioridad.MEDIA))

    # 3. Registrar Vehículos Coordinados
    for v in estado_inicial.vehiculos:
        await AgenteVehiculoCoordinado.register(runtime, type=v.id, factory=lambda v_data=v: AgenteVehiculoCoordinado(v_data.id, v_data.capacidad_max, v_data.coste_paso, estado_inicial.penalizacion_trafico, v_data.posicion))
        await runtime.add_subscription(TypePrefixSubscription(topic_type_prefix="default", agent_type=v.id))
        
    # 4. Registrar Coordinador
    info_depositos = {d.id: d.posicion for d in estado_inicial.depositos}
    await AgenteCoordinador.register(runtime, type="coordinador", factory=lambda: AgenteCoordinador("coordinador", len(estado_inicial.clientes), len(estado_inicial.vehiculos), info_depositos, 17, 12))

    runtime.start()
    await ejecutar_simulacion_coordinada(runtime, estado_inicial, AgentId("coordinador", "default"))
    await runtime.stop()

In [ ]:
await setup_y_ejecutar()

🚀 FASE 1: INICIANDO SUBASTAS Y ASIGNACIÓN...
📢 Coordinador: Iniciando fase de descubrimiento de demanda...
📦 Coordinador: Todas las peticiones recibidas. Ordenando por prioridad...
🏆 Ganador para subasta sub_C1_D2: V1
   (Coste marginal: 0.0, Espacio restante: 0)
🏆 V1 ganó sub_C1_D2 (Inserción en pos 0 y 1)
🏆 Ganador para subasta sub_C4_D2: V2
   (Coste marginal: 0.0, Espacio restante: 0)
🏆 V2 ganó sub_C4_D2 (Inserción en pos 0 y 1)
🏆 Ganador para subasta sub_C1_D1: V3
   (Coste marginal: 0.0, Espacio restante: 15)
🏆 V3 ganó sub_C1_D1 (Inserción en pos 0 y 1)
🏆 Ganador para subasta sub_C4_D1: V3
   (Coste marginal: 0.0, Espacio restante: 0)
🏆 V3 ganó sub_C4_D1 (Inserción en pos 0 y 1)
⚠️ Ningún vehículo pudo asumir la tarea sub_C3_D1
⚠️ Ningún vehículo pudo asumir la tarea sub_C3_D2
⚠️ Ningún vehículo pudo asumir la tarea sub_C5_D1
⚠️ Ningún vehículo pudo asumir la tarea sub_C5_D2
⚠️ Ningún vehículo pudo asumir la tarea sub_C6_D1
⚠️ Ningún vehículo pudo asumir la tarea sub_C2_D2
⚠️ Nin